In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import pandas as pd

file_path = "/kaggle/input/datasets/razanihababdellatif/womens-e-commerce-clothing-reviews-and-sales/women_clothing_50k.csv"

df = pd.read_csv(file_path)

df.head()

,product_id,category,subcategory,material,color,size,season,style,price_usd,discount_percent,rating,review_count,stock_quantity,units_sold,return_rate
0,WC000001,Jeans,Skinny Jeans,Denim,Red,M,Summer,Business Casual,37.00,0,0.0,0,2,14,0.17
1,WC000002,Dresses,Wrap Dress,Cotton,White,M,Winter,Casual,59.12,10,0.0,0,13,31,0.13
2,WC000003,T-Shirts,V-Neck T-Shirt,Cotton,Purple,M,All Season,Casual,27.65,0,0.0,0,10,2,0.11
3,WC000004,Sweaters,Oversized Sweater,Wool,Brown,S,Winter,Formal,72.64,0,2.9,17,104,292,0.14
4,WC000005,Sweaters,Oversized Sweater,Wool,White,M,Winter,Minimalist,72.61,40,0.0,0,12,15,0.07


In [2]:
df.columns.tolist()

['product_id',
 'category',
 'subcategory',
 'material',
 'color',
 'size',
 'season',
 'style',
 'price_usd',
 'discount_percent',
 'rating',
 'review_count',
 'stock_quantity',
 'units_sold',
 'return_rate']

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        50000 non-null  object 
 1   category          50000 non-null  object 
 2   subcategory       50000 non-null  object 
 3   material          50000 non-null  object 
 4   color             50000 non-null  object 
 5   size              50000 non-null  object 
 6   season            50000 non-null  object 
 7   style             50000 non-null  object 
 8   price_usd         50000 non-null  float64
 9   discount_percent  50000 non-null  int64  
 10  rating            50000 non-null  float64
 11  review_count      50000 non-null  int64  
 12  stock_quantity    50000 non-null  int64  
 13  units_sold        50000 non-null  int64  
 14  return_rate       50000 non-null  float64
dtypes: float64(3), int64(4), object(8)
memory usage: 5.7+ MB


In [13]:
features = [
    'category',
    'subcategory',
    'material',
    'color',
    'season',
    'style'
]

df['combined_features'] = (
    df['category'] + ' ' +
    df['category'] + ' ' +
    df['subcategory'] + ' ' +
    df['subcategory'] + ' ' +
    df['subcategory'] + ' ' +
    df['material'] + ' ' +
    df['color'] + ' ' +
    df['season'] + ' ' +
    df['style']
)
df[['product_id', 'combined_features']].head()

,product_id,combined_features
0,WC000001,Jeans Jeans Skinny Jeans Skinny Jeans Skinny J...
1,WC000002,Dresses Dresses Wrap Dress Wrap Dress Wrap Dre...
2,WC000003,T-Shirts T-Shirts V-Neck T-Shirt V-Neck T-Shir...
3,WC000004,Sweaters Sweaters Oversized Sweater Oversized ...
4,WC000005,Sweaters Sweaters Oversized Sweater Oversized ...


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

feature_matrix = vectorizer.fit_transform(
    df['combined_features']
)

print(feature_matrix.shape)

(50000, 108)


In [17]:
from sklearn.neighbors import NearestNeighbors

In [18]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=6
)

model.fit(feature_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

In [19]:
def recommend(product_id, n_recommendations=5):

    # Trouver l'index du produit
    matches = df.index[df['product_id'] == product_id]

    if len(matches) == 0:
        print("Produit non trouvé.")
        return None

    product_index = matches[0]

    # Trouver les voisins les plus proches
    distances, indices = model.kneighbors(
        feature_matrix[product_index],
        n_neighbors=n_recommendations + 1
    )

    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]

    # Résultat
    recommendations = df.iloc[recommended_indices][[
        'product_id',
        'category',
        'subcategory',
        'material',
        'color',
        'season',
        'style',
        'size',
        'price_usd',
        'rating',
        'return_rate'
    ]].copy()

    recommendations['similarity_score'] = 1 - recommended_distances

    return recommendations

In [20]:
df[['product_id', 'category', 'subcategory']].head(10)

,product_id,category,subcategory
0,WC000001,Jeans,Skinny Jeans
1,WC000002,Dresses,Wrap Dress
2,WC000003,T-Shirts,V-Neck T-Shirt
3,WC000004,Sweaters,Oversized Sweater
4,WC000005,Sweaters,Oversized Sweater
5,WC000006,Dresses,Mini Dress
6,WC000007,Blouses,Button-Up Blouse
7,WC000008,Dresses,Shirt Dress
8,WC000009,Jumpsuits,Utility Jumpsuit
9,WC000010,Dresses,Shirt Dress


In [21]:
recommend('PROD_00001')

Produit non trouvé.


In [25]:
recommend("WC000001")

,product_id,category,subcategory,material,color,season,style,size,price_usd,rating,return_rate,similarity_score
44628,WC044629,Jeans,Skinny Jeans,Denim,Red,Summer,Business Casual,M,98.46,4.7,0.13,1.000000
33318,WC033319,Jeans,Skinny Jeans,Denim,Red,Summer,Casual,M,50.36,3.8,0.10,0.992399
41747,WC041748,Jeans,Skinny Jeans,Denim,Red,Summer,Casual,L,84.64,3.8,0.11,0.992399
20576,WC020577,Jeans,Skinny Jeans,Denim,Red,Autumn,Business Casual,XS,93.74,5.0,0.13,0.987888
11814,WC011815,Jeans,Skinny Jeans,Denim,Red,Winter,Business Casual,S,62.41,5.0,0.12,0.987716


In [27]:
df[df['product_id'] == "WC000001"]

,product_id,category,subcategory,material,color,size,season,style,price_usd,discount_percent,rating,review_count,stock_quantity,units_sold,return_rate,combined_features
0,WC000001,Jeans,Skinny Jeans,Denim,Red,M,Summer,Business Casual,37.0,0,0.0,0,2,14,0.17,Jeans Jeans Skinny Jeans Skinny Jeans Skinny J...


In [28]:
df.iloc[0]

product_id                                                    WC000001
category                                                         Jeans
subcategory                                               Skinny Jeans
material                                                         Denim
color                                                              Red
size                                                                 M
season                                                          Summer
style                                                  Business Casual
price_usd                                                         37.0
discount_percent                                                     0
rating                                                             0.0
review_count                                                         0
stock_quantity                                                       2
units_sold                                                          14
return

In [29]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors

In [30]:
features = [
    'category',
    'subcategory',
    'material',
    'color',
    'season',
    'style'
]

encoder = OneHotEncoder(handle_unknown='ignore')

feature_matrix = encoder.fit_transform(df[features])

print(feature_matrix.shape)

(50000, 105)


In [31]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=10
)

model.fit(feature_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=10)

In [ ]:
def recommend(product_id, n_recommendations=5):

    # Trouver le produit
    matches = df.index[df['product_id'] == product_id]

    if len(matches) == 0:
        print("Produit non trouvé.")
        return None

    product_index = matches[0]

    # Trouver les voisins similaires
    distances, indices = model.kneighbors(
        feature_matrix[product_index],
        n_neighbors=n_recommendations + 1
    )

    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]

    recommendations = df.iloc[recommended_indices][[
        'product_id',
        'category',
        'subcategory',
        'material',
        'color',
        'size',
        'season',
        'style',
        'price_usd',
        'rating',
        'units_sold',
        'return_rate'
    ]].copy()

    recommendations['similarity_score'] = (
        1 - recommended_distances
    )

    return recommendations.sort_values(
        'similarity_score',
        ascending=False
    )

In [32]:
recommend("WC000001")

,product_id,category,subcategory,material,color,season,style,size,price_usd,rating,return_rate,similarity_score
44628,WC044629,Jeans,Skinny Jeans,Denim,Red,Summer,Business Casual,M,98.46,4.7,0.13,1.000000
35482,WC035483,Jeans,Skinny Jeans,Denim,Navy,Summer,Business Casual,L,85.86,3.1,0.21,0.833333
49946,WC049947,Jeans,Skinny Jeans,Denim,White,Summer,Business Casual,M,68.14,2.5,0.16,0.833333
35581,WC035582,Jeans,Skinny Jeans,Denim,Navy,Summer,Business Casual,M,73.97,0.0,0.00,0.833333
35208,WC035209,Jeans,Skinny Jeans,Denim,Red,Summer,Minimalist,M,94.01,0.0,0.15,0.833333


In [49]:
import numpy as np

def recommend_hybrid(product_id, n_recommendations=5):
    # Trouver le produit (position réelle dans df, pas le label)
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]
    target = df.iloc[product_index]

    # On cherche plus de candidats avant de les reclasser
    n_query = max(100, n_recommendations + 1)
    distances, indices = model.kneighbors(
        feature_matrix[product_index].reshape(1, -1),
        n_neighbors=n_query
    )
    indices = indices.flatten()
    distances = distances.flatten()

    # Enlever le produit lui-même
    mask = indices != product_index
    indices = indices[mask]
    distances = distances[mask]

    recommendations = df.iloc[indices].copy()

    # Similarité basée sur les caractéristiques
    recommendations["content_similarity"] = 1 - distances

    # Similarité basée sur le prix (relative au prix du produit cible,
    # pas au range global du catalogue)
    target_price = target["price_usd"]
    recommendations["price_similarity"] = 1 - (
        abs(recommendations["price_usd"] - target_price)
        / max(target_price, 1)
    )
    recommendations["price_similarity"] = recommendations[
        "price_similarity"
    ].clip(lower=0)

    # Score final
    recommendations["final_score"] = (
        0.75 * recommendations["content_similarity"]
        + 0.25 * recommendations["price_similarity"]
    )

    recommendations = recommendations.sort_values(
        "final_score", ascending=False
    ).head(n_recommendations)

    return recommendations[[
        "product_id", "category", "subcategory", "material", "color",
        "season", "style", "size", "price_usd", "rating", "return_rate",
        "content_similarity", "price_similarity", "final_score"
    ]]
  

In [50]:
recommend_hybrid("WC000001")

,product_id,category,subcategory,material,color,season,style,size,price_usd,rating,return_rate,content_similarity,price_similarity,final_score
35477,WC035478,Jeans,Skinny Jeans,Denim,Navy,Summer,Business Casual,L,36.17,3.5,0.14,0.833333,0.977568,0.869392
46537,WC046538,Jeans,Skinny Jeans,Denim,Navy,Summer,Business Casual,M,35.79,0.0,0.16,0.833333,0.967297,0.866824
40777,WC040778,Jeans,Skinny Jeans,Denim,Pink,Summer,Business Casual,XS,34.57,4.1,0.11,0.833333,0.934324,0.858581
28618,WC028619,Jeans,Skinny Jeans,Denim,Navy,Summer,Business Casual,S,39.99,3.4,0.22,0.833333,0.919189,0.854797
40723,WC040724,Jeans,Skinny Jeans,Denim,Navy,Summer,Business Casual,L,40.62,4.5,0.12,0.833333,0.902162,0.850541


In [51]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------
# 1. Préparation des features
# ---------------------------------------------------------
cat_features = ['category', 'subcategory', 'material', 'color', 'season', 'style', 'size']
num_features = ['price_usd', 'discount_percent', 'rating', 'return_rate']

encoders = {}
cat_data = {}
cat_dims = {}

for col in cat_features:
    le = LabelEncoder()
    cat_data[col] = le.fit_transform(df[col])
    encoders[col] = le
    cat_dims[col] = len(le.classes_)

scaler = StandardScaler()
num_data = scaler.fit_transform(df[num_features])

# ---------------------------------------------------------
# 2. Modèle : Embeddings catégoriels + features numériques -> Dense -> embedding produit
# ---------------------------------------------------------
EMBED_DIM_CAT = 8      # dimension des embeddings par variable catégorielle
PRODUCT_EMBED_DIM = 32  # dimension finale de l'embedding produit

cat_inputs = []
cat_embeds = []

for col in cat_features:
    inp = layers.Input(shape=(1,), name=f"input_{col}")
    emb = layers.Embedding(
        input_dim=cat_dims[col],
        output_dim=EMBED_DIM_CAT,
        name=f"embed_{col}"
    )(inp)
    emb = layers.Flatten()(emb)
    cat_inputs.append(inp)
    cat_embeds.append(emb)

num_input = layers.Input(shape=(len(num_features),), name="input_numerical")

x = layers.Concatenate()(cat_embeds + [num_input])
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
product_embedding = layers.Dense(
    PRODUCT_EMBED_DIM, activation=None, name="product_embedding"
)(x)

# Auto-encodeur : on force le modèle à reconstruire les features numériques
# à partir de l'embedding, pour que l'embedding soit informatif (pas besoin
# de labels externes = apprentissage non supervisé)
reconstruction = layers.Dense(len(num_features), activation=None, name="reconstruction")(product_embedding)

embedding_model = Model(inputs=cat_inputs + [num_input], outputs=product_embedding)
training_model = Model(inputs=cat_inputs + [num_input], outputs=reconstruction)

training_model.compile(optimizer="adam", loss="mse")

# ---------------------------------------------------------
# 3. Entraînement
# ---------------------------------------------------------
train_inputs = [cat_data[col].reshape(-1, 1) for col in cat_features] + [num_data]

training_model.fit(
    train_inputs,
    num_data,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

# ---------------------------------------------------------
# 4. Génération des embeddings produits
# ---------------------------------------------------------
product_embeddings = embedding_model.predict(train_inputs, batch_size=512)
print(product_embeddings.shape)  # (n_produits, PRODUCT_EMBED_DIM)

# ---------------------------------------------------------
# 5. Recommandation par cosine similarity sur les embeddings
# ---------------------------------------------------------
def recommend_nn(product_id, n_recommendations=5):
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]

    target_vec = product_embeddings[product_index].reshape(1, -1)
    sims = cosine_similarity(target_vec, product_embeddings).flatten()

    # Exclure le produit lui-même
    sims[product_index] = -np.inf

    top_indices = np.argsort(sims)[::-1][:n_recommendations]

    recommendations = df.iloc[top_indices][[
        'product_id', 'category', 'subcategory', 'material', 'color',
        'season', 'style', 'size', 'price_usd', 'rating', 'return_rate'
    ]].copy()
    recommendations['similarity_score'] = sims[top_indices]

    return recommendations.reset_index(drop=True)

recommend_nn("WC000001")

I0000 00:00:1786699461.583187      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786699461.586277      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/20
 61/176 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5003

I0000 00:00:1786699468.070563     205 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


176/176 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - loss: 0.0988 - val_loss: 0.0020
Epoch 2/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0183 - val_loss: 0.0013
Epoch 3/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0148 - val_loss: 0.0013
Epoch 4/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0135 - val_loss: 0.0012
Epoch 5/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0124 - val_loss: 0.0012
Epoch 6/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0116 - val_loss: 0.0010
Epoch 7/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0107 - val_loss: 0.0021
Epoch 8/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0100 - val_loss: 0.0018
Epoch 9/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0094 - val_loss: 0.0022
Epoch 10/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0087 - val_loss: 0.0021
Epoch 11/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0081 - val_loss: 0.0023
Epoch 12/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0

,product_id,category,subcategory,material,color,season,style,size,price_usd,rating,return_rate,similarity_score
0,WC035870,Dresses,Maxi Dress,Viscose,Black,All Season,Streetwear,XXL,37.81,0.0,0.17,0.999924
1,WC027704,Jeans,Bootcut Jeans,Denim,Green,Spring,Elegant,S,38.77,0.0,0.17,0.999863
2,WC032296,Jeans,Straight Jeans,Denim,Purple,Spring,Business Casual,XL,35.18,0.0,0.17,0.999837
3,WC019726,Jeans,Wide-Leg Jeans,Denim,Gray,Spring,Bohemian,M,34.94,0.0,0.17,0.999746
4,WC006809,Dresses,Bodycon Dress,Cotton,Black,Summer,Streetwear,M,34.42,0.0,0.17,0.999690


In [52]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, callbacks
from sklearn.metrics.pairwise import cosine_similarity

# ===========================================================
# 1. Préparation des features (identique à avant)
# ===========================================================
cat_features = ['category', 'subcategory', 'material', 'color', 'season', 'style', 'size']
num_features = ['price_usd', 'discount_percent', 'rating', 'return_rate']

encoders = {}
cat_data = {}
cat_dims = {}

for col in cat_features:
    le = LabelEncoder()
    cat_data[col] = le.fit_transform(df[col])
    encoders[col] = le
    cat_dims[col] = len(le.classes_)

scaler = StandardScaler()
num_data = scaler.fit_transform(df[num_features])

all_inputs = [cat_data[col].reshape(-1, 1) for col in cat_features] + [num_data]
n_samples = len(df)

# Split train/val pour l'EarlyStopping
idx_train, idx_val = train_test_split(
    np.arange(n_samples), test_size=0.15, random_state=42
)

def subset(inputs, idx):
    return [arr[idx] for arr in inputs]

train_inputs = subset(all_inputs, idx_train)
val_inputs = subset(all_inputs, idx_val)
train_target = num_data[idx_train]
val_target = num_data[idx_val]

# ===========================================================
# 2. Autoencoder amélioré : bottleneck contraint + régularisation
# ===========================================================
EMBED_DIM_CAT = 8
BOTTLENECK_DIM = 16     # plus petit -> force une compression plus forte
L2_REG = 1e-4
DROPOUT_RATE = 0.3

cat_inputs = []
cat_embeds = []

for col in cat_features:
    inp = layers.Input(shape=(1,), name=f"input_{col}")
    emb = layers.Embedding(
        input_dim=cat_dims[col],
        output_dim=EMBED_DIM_CAT,
        embeddings_regularizer=regularizers.l2(L2_REG),
        name=f"embed_{col}"
    )(inp)
    emb = layers.Flatten()(emb)
    cat_inputs.append(inp)
    cat_embeds.append(emb)

num_input = layers.Input(shape=(len(num_features),), name="input_numerical")

# --- Encodeur ---
x = layers.Concatenate()(cat_embeds + [num_input])
x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)

x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)

# --- Bottleneck (l'embedding produit final) ---
bottleneck = layers.Dense(
    BOTTLENECK_DIM,
    activation=None,
    activity_regularizer=regularizers.l1(1e-5),  # encourage un embedding "sparse"/discriminant
    name="product_embedding"
)(x)

# --- Décodeur ---
d = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(bottleneck)
d = layers.Dropout(DROPOUT_RATE)(d)
d = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(d)
reconstruction = layers.Dense(len(num_features), activation=None, name="reconstruction")(d)

embedding_model = Model(inputs=cat_inputs + [num_input], outputs=bottleneck)
autoencoder = Model(inputs=cat_inputs + [num_input], outputs=reconstruction)

autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

# ===========================================================
# 3. Entraînement avec EarlyStopping
# ===========================================================
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

history = autoencoder.fit(
    train_inputs,
    train_target,
    validation_data=(val_inputs, val_target),
    epochs=100,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ===========================================================
# 4. Génération des embeddings produits (sur tout le dataset)
# ===========================================================
product_embeddings_ae = embedding_model.predict(all_inputs, batch_size=512)
print("Embedding shape:", product_embeddings_ae.shape)

# ===========================================================
# 5. Recommandation par cosine similarity (Deep Learning)
# ===========================================================
def recommend_autoencoder(product_id, n_recommendations=5):
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]

    target_vec = product_embeddings_ae[product_index].reshape(1, -1)
    sims = cosine_similarity(target_vec, product_embeddings_ae).flatten()
    sims[product_index] = -np.inf

    top_indices = np.argsort(sims)[::-1][:n_recommendations]

    recommendations = df.iloc[top_indices][[
        'product_id', 'category', 'subcategory', 'material', 'color',
        'season', 'style', 'size', 'price_usd', 'rating', 'return_rate'
    ]].copy()
    recommendations['similarity_score'] = sims[top_indices]

    return recommendations.reset_index(drop=True)

# ===========================================================
# 6. Comparaison expérimentale : Hybrid KNN vs Autoencoder
# ===========================================================
def compare_recommendations(product_id, n_recommendations=5):
    print(f"=== Cible : {product_id} ===\n")

    print("--- Hybrid KNN (classique) ---")
    knn_res = recommend_hybrid(product_id, n_recommendations)
    print(knn_res[['product_id', 'category', 'subcategory', 'color', 'price_usd', 'final_score']])

    print("\n--- Autoencoder (Deep Learning) ---")
    ae_res = recommend_autoencoder(product_id, n_recommendations)
    print(ae_res[['product_id', 'category', 'subcategory', 'color', 'price_usd', 'similarity_score']])

    # Taux de recouvrement entre les deux listes
    overlap = set(knn_res['product_id']) & set(ae_res['product_id'])
    print(f"\nRecouvrement : {len(overlap)}/{n_recommendations} produits communs -> {overlap}")

    return knn_res, ae_res

compare_recommendations("WC000001")

Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - loss: 0.3350 - val_loss: 0.5282 - learning_rate: 0.0010
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1980 - val_loss: 0.2549 - learning_rate: 0.0010
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1669 - val_loss: 0.1379 - learning_rate: 0.0010
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1464 - val_loss: 0.0871 - learning_rate: 0.0010
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1328 - val_loss: 0.0784 - learning_rate: 0.0010
Epoch 6/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1200 - val_loss: 0.0654 - learning_rate: 0.0010
Epoch 7/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1109 - val_loss: 0.0659 - learning_rate: 0.0010
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1042 - val_loss: 0.0706 - learning_rate: 0.0010
Epoch 9/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0975 - val_loss: 0.0711 - learning_rate: 0.0010

(      product_id category   subcategory material color  season  \
 35477   WC035478    Jeans  Skinny Jeans    Denim  Navy  Summer   
 46537   WC046538    Jeans  Skinny Jeans    Denim  Navy  Summer   
 40777   WC040778    Jeans  Skinny Jeans    Denim  Pink  Summer   
 28618   WC028619    Jeans  Skinny Jeans    Denim  Navy  Summer   
 40723   WC040724    Jeans  Skinny Jeans    Denim  Navy  Summer   
 
                  style size  price_usd  rating  return_rate  \
 35477  Business Casual    L      36.17     3.5         0.14   
 46537  Business Casual    M      35.79     0.0         0.16   
 40777  Business Casual   XS      34.57     4.1         0.11   
 28618  Business Casual    S      39.99     3.4         0.22   
 40723  Business Casual    L      40.62     4.5         0.12   
 
        content_similarity  price_similarity  final_score  
 35477            0.833333          0.977568     0.869392  
 46537            0.833333          0.967297     0.866824  
 40777            0.833333    

In [53]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, callbacks
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# ===========================================================
# 1. Préparation des features (identique)
# ===========================================================
cat_features = ['category', 'subcategory', 'material', 'color', 'season', 'style', 'size']
num_features = ['price_usd', 'discount_percent', 'rating', 'return_rate']

encoders = {}
cat_data = {}
cat_dims = {}

for col in cat_features:
    le = LabelEncoder()
    cat_data[col] = le.fit_transform(df[col])
    encoders[col] = le
    cat_dims[col] = len(le.classes_)

scaler = StandardScaler()
num_data = scaler.fit_transform(df[num_features]).astype("float32")

n_samples = len(df)

# ===========================================================
# 2. Génération de triplets (anchor, positive, negative)
# ===========================================================
# Contrairement à l'autoencoder (tâche de reconstruction), ici on entraîne
# l'embedding directement sur une notion de similarité produit :
#   - positive = même category + subcategory (variante du même produit)
#   - negative = category différente (produit clairement différent)
# C'est un signal métier explicite, pas juste une compression de features.

rng = np.random.default_rng(42)

category_arr = df['category'].values
subcategory_arr = df['subcategory'].values

# Index des produits groupés par (category, subcategory) pour piocher des positifs
group_key = pd.Series(list(zip(category_arr, subcategory_arr)))
group_to_indices = group_key.groupby(group_key).apply(lambda s: s.index.to_numpy())

def build_triplets(n_triplets):
    anchors = rng.integers(0, n_samples, size=n_triplets)
    positives = np.empty(n_triplets, dtype=int)
    negatives = np.empty(n_triplets, dtype=int)

    for i, a in enumerate(anchors):
        key = (category_arr[a], subcategory_arr[a])
        same_group = group_to_indices[key]
        # positif : même groupe, différent du anchor si possible
        if len(same_group) > 1:
            p = rng.choice(same_group)
            while p == a:
                p = rng.choice(same_group)
        else:
            p = a
        positives[i] = p

        # négatif : catégorie différente
        n = rng.integers(0, n_samples)
        while category_arr[n] == category_arr[a]:
            n = rng.integers(0, n_samples)
        negatives[i] = n

    return anchors, positives, negatives

N_TRIPLETS = 100_000
anchor_idx, pos_idx, neg_idx = build_triplets(N_TRIPLETS)

# ===========================================================
# 3. Modèle d'embedding partagé (tour unique réutilisée 3x)
# ===========================================================
EMBED_DIM_CAT = 8
PRODUCT_EMBED_DIM = 32
L2_REG = 1e-4
DROPOUT_RATE = 0.2
MARGIN = 0.3  # marge du triplet loss

def build_embedding_tower():
    cat_inputs = [layers.Input(shape=(1,), name=f"input_{col}") for col in cat_features]
    cat_embeds = []
    for inp, col in zip(cat_inputs, cat_features):
        emb = layers.Embedding(
            input_dim=cat_dims[col],
            output_dim=EMBED_DIM_CAT,
            embeddings_regularizer=regularizers.l2(L2_REG),
            name=f"embed_{col}"
        )(inp)
        cat_embeds.append(layers.Flatten()(emb))

    num_input = layers.Input(shape=(len(num_features),), name="input_numerical")

    x = layers.Concatenate()(cat_embeds + [num_input])
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)

    embedding = layers.Dense(PRODUCT_EMBED_DIM, activation=None, name="product_embedding")(x)
    embedding = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1), name="l2_normalize"
    )(embedding)

    return Model(inputs=cat_inputs + [num_input], outputs=embedding, name="embedding_tower")

embedding_tower = build_embedding_tower()

# ===========================================================
# 4. Modèle triplet (3 branches partageant les mêmes poids)
# ===========================================================
def make_input_set(prefix):
    return [layers.Input(shape=(1,), name=f"{prefix}_{col}") for col in cat_features] + \
           [layers.Input(shape=(len(num_features),), name=f"{prefix}_numerical")]

anchor_inputs = make_input_set("anchor")
positive_inputs = make_input_set("positive")
negative_inputs = make_input_set("negative")

anchor_emb = embedding_tower(anchor_inputs)
positive_emb = embedding_tower(positive_inputs)
negative_emb = embedding_tower(negative_inputs)

merged = layers.Concatenate(axis=1)([anchor_emb, positive_emb, negative_emb])
triplet_model = Model(
    inputs=anchor_inputs + positive_inputs + negative_inputs,
    outputs=merged
)

def triplet_loss(y_true, y_pred, embed_dim=PRODUCT_EMBED_DIM, margin=MARGIN):
    a = y_pred[:, :embed_dim]
    p = y_pred[:, embed_dim:2*embed_dim]
    n = y_pred[:, 2*embed_dim:]
    pos_dist = tf.reduce_sum(tf.square(a - p), axis=1)
    neg_dist = tf.reduce_sum(tf.square(a - n), axis=1)
    loss = tf.maximum(pos_dist - neg_dist + margin, 0.0)
    return tf.reduce_mean(loss)

triplet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=triplet_loss)

# ===========================================================
# 5. Préparation des tenseurs d'entrée pour les triplets
# ===========================================================
def gather_inputs(indices):
    return [cat_data[col][indices].reshape(-1, 1) for col in cat_features] + [num_data[indices]]

train_data = (
    gather_inputs(anchor_idx) + gather_inputs(pos_idx) + gather_inputs(neg_idx)
)
dummy_y = np.zeros((N_TRIPLETS, 1))  # y_true inutilisé, la loss dépend seulement de y_pred

# ===========================================================
# 6. Entraînement avec EarlyStopping
# ===========================================================
early_stop = callbacks.EarlyStopping(monitor="loss", patience=4, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5, patience=2, min_lr=1e-5)

history = triplet_model.fit(
    train_data,
    dummy_y,
    epochs=30,
    batch_size=512,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ===========================================================
# 7. Génération des embeddings produits finaux
# ===========================================================
all_inputs = [cat_data[col].reshape(-1, 1) for col in cat_features] + [num_data]
product_embeddings_nn = embedding_tower.predict(all_inputs, batch_size=1024)
print("Embedding shape:", product_embeddings_nn.shape)

# ===========================================================
# 8. Recommandation par cosine similarity (Neural Embedding)
# ===========================================================
def recommend_neural_embedding(product_id, n_recommendations=5):
    matches = np.flatnonzero(df['product_id'].values == product_id)
    if len(matches) == 0:
        print("Produit non trouvé.")
        return None
    product_index = matches[0]

    target_vec = product_embeddings_nn[product_index].reshape(1, -1)
    sims = cosine_similarity(target_vec, product_embeddings_nn).flatten()
    sims[product_index] = -np.inf

    top_indices = np.argsort(sims)[::-1][:n_recommendations]

    recommendations = df.iloc[top_indices][[
        'product_id', 'category', 'subcategory', 'material', 'color',
        'season', 'style', 'size', 'price_usd', 'rating', 'return_rate'
    ]].copy()
    recommendations['similarity_score'] = sims[top_indices]

    return recommendations.reset_index(drop=True)

# ===========================================================
# 9. Comparaison à 3 : Hybrid KNN vs Autoencoder vs Neural Embedding
# ===========================================================
def compare_all(product_id, n_recommendations=5):
    print(f"=== Cible : {product_id} ===\n")

    knn_res = recommend_hybrid(product_id, n_recommendations)
    ae_res = recommend_autoencoder(product_id, n_recommendations)
    nn_res = recommend_neural_embedding(product_id, n_recommendations)

    print("--- Hybrid KNN ---")
    print(knn_res[['product_id', 'subcategory', 'color', 'price_usd']])
    print("\n--- Autoencoder ---")
    print(ae_res[['product_id', 'subcategory', 'color', 'price_usd']])
    print("\n--- Neural Embedding (triplet loss) ---")
    print(nn_res[['product_id', 'subcategory', 'color', 'price_usd']])

    knn_set, ae_set, nn_set = set(knn_res['product_id']), set(ae_res['product_id']), set(nn_res['product_id'])
    print(f"\nRecouvrement KNN ∩ AE  : {knn_set & ae_set}")
    print(f"Recouvrement KNN ∩ NN  : {knn_set & nn_set}")
    print(f"Recouvrement AE  ∩ NN  : {ae_set & nn_set}")

    return knn_res, ae_res, nn_res

compare_all("WC000001")

Epoch 1/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - loss: 0.0952 - learning_rate: 0.0010
Epoch 2/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0176 - learning_rate: 0.0010
Epoch 3/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0133 - learning_rate: 0.0010
Epoch 4/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0105 - learning_rate: 0.0010
Epoch 5/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0085 - learning_rate: 0.0010
Epoch 6/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0069 - learning_rate: 0.0010
Epoch 7/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0055 - learning_rate: 0.0010
Epoch 8/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0045 - learning_rate: 0.0010
Epoch 9/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0036 - learning_rate: 0.0010
Epoch 10/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0030 - learning_rate: 0.0010
Epoch 11/30
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0024 - learning_rate: 0.

(      product_id category   subcategory material color  season  \
 35477   WC035478    Jeans  Skinny Jeans    Denim  Navy  Summer   
 46537   WC046538    Jeans  Skinny Jeans    Denim  Navy  Summer   
 40777   WC040778    Jeans  Skinny Jeans    Denim  Pink  Summer   
 28618   WC028619    Jeans  Skinny Jeans    Denim  Navy  Summer   
 40723   WC040724    Jeans  Skinny Jeans    Denim  Navy  Summer   
 
                  style size  price_usd  rating  return_rate  \
 35477  Business Casual    L      36.17     3.5         0.14   
 46537  Business Casual    M      35.79     0.0         0.16   
 40777  Business Casual   XS      34.57     4.1         0.11   
 28618  Business Casual    S      39.99     3.4         0.22   
 40723  Business Casual    L      40.62     4.5         0.12   
 
        content_similarity  price_similarity  final_score  
 35477            0.833333          0.977568     0.869392  
 46537            0.833333          0.967297     0.866824  
 40777            0.833333    

In [55]:

# ===========================================================
# FIX 1 — Autoencoder : reconstruire AUSSI le catégoriel
# ===========================================================
d = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(bottleneck)
d = layers.Dropout(DROPOUT_RATE)(d)
d = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(d)

# une tête softmax par variable catégorielle + une tête numérique
outputs = []
losses = {}
loss_weights = {}

for col in cat_features:
    out = layers.Dense(cat_dims[col], activation="softmax", name=f"recon_{col}")(d)
    outputs.append(out)
    losses[f"recon_{col}"] = "sparse_categorical_crossentropy"
    loss_weights[f"recon_{col}"] = 1.0

num_out = layers.Dense(len(num_features), activation=None, name="recon_numerical")(d)
outputs.append(num_out)
losses["recon_numerical"] = "mse"
loss_weights["recon_numerical"] = 1.0  # à ajuster : le num pèse sinon trop peu vs 7 têtes catégorielles

embedding_model = Model(inputs=cat_inputs + [num_input], outputs=bottleneck)
autoencoder = Model(inputs=cat_inputs + [num_input], outputs=outputs)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=losses, loss_weights=loss_weights)

train_targets = {f"recon_{col}": cat_data[col][idx_train] for col in cat_features}
train_targets["recon_numerical"] = train_target

val_targets = {f"recon_{col}": cat_data[col][idx_val] for col in cat_features}
val_targets["recon_numerical"] = val_target

autoencoder.fit(train_inputs, train_targets,
                 validation_data=(val_inputs, val_targets),
                 epochs=100, batch_size=256,
                 callbacks=[early_stop, reduce_lr], verbose=1)

Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 22s 59ms/step - loss: 14.5986 - recon_category_loss: 1.7979 - recon_color_loss: 2.3544 - recon_material_loss: 1.7841 - recon_numerical_loss: 0.3569 - recon_season_loss: 1.4797 - recon_size_loss: 1.6571 - recon_style_loss: 1.8521 - recon_subcategory_loss: 3.2592 - val_loss: 11.4863 - val_recon_category_loss: 0.8431 - val_recon_color_loss: 2.2931 - val_recon_material_loss: 1.3085 - val_recon_numerical_loss: 0.2867 - val_recon_season_loss: 1.2310 - val_recon_size_loss: 1.5598 - val_recon_style_loss: 1.5755 - val_recon_subcategory_loss: 2.3327 - learning_rate: 0.0010
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 10.4810 - recon_category_loss: 0.5491 - recon_color_loss: 2.3314 - recon_material_loss: 1.2626 - recon_numerical_loss: 0.4834 - recon_season_loss: 1.0253 - recon_size_loss: 1.4215 - recon_style_loss: 1.3895 - recon_subcategory_loss: 1.9988 - val_loss: 8.5291 - val_recon_category_loss: 0.1778 - val_recon_color_loss: 2.3007 - va

In [60]:
# ===========================================================
# FIX 2 — Triplet : négatifs durs (même category, autre subcategory)
# ===========================================================
subcat_group_key = pd.Series(list(zip(category_arr, subcategory_arr)))
category_series = pd.Series(category_arr)
category_to_indices = category_series.groupby(category_series).apply(lambda s: s.index.to_numpy())

def build_triplets_hard(n_triplets, hard_ratio=0.7):
    anchors = rng.integers(0, n_samples, size=n_triplets)
    positives = np.empty(n_triplets, dtype=int)
    negatives = np.empty(n_triplets, dtype=int)

    for i, a in enumerate(anchors):
        key = (category_arr[a], subcategory_arr[a])
        same_group = group_to_indices[key]
        p = rng.choice(same_group)
        while p == a and len(same_group) > 1:
            p = rng.choice(same_group)
        positives[i] = p

        if rng.random() < hard_ratio:
            # négatif dur : même category, autre subcategory
            same_cat = category_to_indices[category_arr[a]]
            n = rng.choice(same_cat)
            tries = 0
            while subcategory_arr[n] == subcategory_arr[a] and tries < 20:
                n = rng.choice(same_cat)
                tries += 1
        else:
            # négatif facile : autre category (garde un peu de diversité de gradient)
            n = rng.integers(0, n_samples)
            while category_arr[n] == category_arr[a]:
                n = rng.integers(0, n_samples)
        negatives[i] = n

    return anchors, positives, negatives

anchor_idx, pos_idx, neg_idx = build_triplets_hard(150_000, hard_ratio=0.7)
# régénérer train_data avec ces nouveaux triplets, réentraîner

In [61]:
def check_embedding_health(embeddings, name):
    norms = np.linalg.norm(embeddings, axis=1)
    pairwise_sample = cosine_similarity(embeddings[:500])
    off_diag = pairwise_sample[~np.eye(500, dtype=bool)]
    print(f"[{name}] std par dimension (moyenne): {embeddings.std(axis=0).mean():.4f}")
    print(f"[{name}] similarité cosine moyenne (paires aléatoires): {off_diag.mean():.4f}")
    print(f"[{name}] similarité cosine std: {off_diag.std():.4f}")
    # proche de 1.0 en moyenne avec std faible = collapse

check_embedding_health(product_embeddings_ae, "Autoencoder")
check_embedding_health(product_embeddings_nn, "Neural Embedding")

[Autoencoder] std par dimension (moyenne): 0.4167
[Autoencoder] similarité cosine moyenne (paires aléatoires): 0.0162
[Autoencoder] similarité cosine std: 0.4567
[Neural Embedding] std par dimension (moyenne): 0.1686
[Neural Embedding] similarité cosine moyenne (paires aléatoires): 0.0380
[Neural Embedding] similarité cosine std: 0.4537


In [62]:
def evaluate_model(recommend_fn, n_products=200, k=5):
    sample_ids = df['product_id'].sample(n_products, random_state=42).values
    cat_match, subcat_match, price_ratios = [], [], []

    for pid in sample_ids:
        target_row = df[df['product_id'] == pid].iloc[0]
        recs = recommend_fn(pid, k)
        if recs is None or len(recs) == 0:
            continue
        cat_match.append((recs['category'] == target_row['category']).mean())
        subcat_match.append((recs['subcategory'] == target_row['subcategory']).mean())
        price_ratios.append((abs(recs['price_usd'] - target_row['price_usd']) / max(target_row['price_usd'], 1)).mean())

    print(f"Category match@{k}: {np.mean(cat_match):.3f}")
    print(f"Subcategory match@{k}: {np.mean(subcat_match):.3f}")
    print(f"Écart prix relatif moyen: {np.mean(price_ratios):.3f}")

print("--- Hybrid KNN ---"); evaluate_model(lambda pid, k: recommend_hybrid(pid, k))
print("--- Autoencoder ---"); evaluate_model(lambda pid, k: recommend_autoencoder(pid, k))
print("--- Neural Embedding ---"); evaluate_model(lambda pid, k: recommend_neural_embedding(pid, k))

--- Hybrid KNN ---
Category match@5: 0.998
Subcategory match@5: 0.928
Écart prix relatif moyen: 0.121
--- Autoencoder ---
Category match@5: 0.262
Subcategory match@5: 0.059
Écart prix relatif moyen: 0.041
--- Neural Embedding ---
Category match@5: 1.000
Subcategory match@5: 0.542
Écart prix relatif moyen: 0.070


In [63]:
# ===========================================================
# A. NEURAL EMBEDDING — marge réduite + négatifs 100% durs
# ===========================================================

MARGIN = 0.1  # réduit de 0.3 -> force une séparation plus fine

def build_triplets_hard_v2(n_triplets):
    anchors = rng.integers(0, n_samples, size=n_triplets)
    positives = np.empty(n_triplets, dtype=int)
    negatives = np.empty(n_triplets, dtype=int)
    skipped_self_positive = 0

    for i, a in enumerate(anchors):
        key = (category_arr[a], subcategory_arr[a])
        same_group = group_to_indices[key]

        if len(same_group) > 1:
            p = rng.choice(same_group)
            tries = 0
            while p == a and tries < 10:
                p = rng.choice(same_group)
                tries += 1
        else:
            # groupe trop petit : on tolère anchor==positive mais on le compte
            p = a
            skipped_self_positive += 1
        positives[i] = p

        # négatif 100% dur : même category, subcategory différente
        same_cat = category_to_indices[category_arr[a]]
        n = rng.choice(same_cat)
        tries = 0
        while subcategory_arr[n] == subcategory_arr[a] and tries < 30:
            n = rng.choice(same_cat)
            tries += 1
        negatives[i] = n

    print(f"Triplets anchor==positive (groupe trop petit): {skipped_self_positive}/{n_triplets}")
    return anchors, positives, negatives

N_TRIPLETS = 200_000  # augmenté pour mieux couvrir les paires de subcategories
anchor_idx, pos_idx, neg_idx = build_triplets_hard_v2(N_TRIPLETS)

# Vérif rapide de couverture par subcategory (diagnostic)
subcat_coverage = pd.Series(subcategory_arr[anchor_idx]).value_counts()
print("Couverture triplets par subcategory (min/max):", subcat_coverage.min(), subcat_coverage.max())

train_data = gather_inputs(anchor_idx) + gather_inputs(pos_idx) + gather_inputs(neg_idx)
dummy_y = np.zeros((N_TRIPLETS, 1))

# Réinitialiser le modèle (nouveaux poids, pas de warm start sur l'ancienne marge)
embedding_tower = build_embedding_tower()
anchor_emb = embedding_tower(anchor_inputs)
positive_emb = embedding_tower(positive_inputs)
negative_emb = embedding_tower(negative_inputs)
merged = layers.Concatenate(axis=1)([anchor_emb, positive_emb, negative_emb])
triplet_model = Model(inputs=anchor_inputs + positive_inputs + negative_inputs, outputs=merged)

def triplet_loss_v2(y_true, y_pred, embed_dim=PRODUCT_EMBED_DIM, margin=MARGIN):
    a = y_pred[:, :embed_dim]
    p = y_pred[:, embed_dim:2*embed_dim]
    n = y_pred[:, 2*embed_dim:]
    pos_dist = tf.reduce_sum(tf.square(a - p), axis=1)
    neg_dist = tf.reduce_sum(tf.square(a - n), axis=1)
    loss = tf.maximum(pos_dist - neg_dist + margin, 0.0)
    return tf.reduce_mean(loss)

triplet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=triplet_loss_v2)

early_stop = callbacks.EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5, patience=2, min_lr=1e-5)

history_nn = triplet_model.fit(
    train_data, dummy_y,
    epochs=40, batch_size=512,
    callbacks=[early_stop, reduce_lr], verbose=1
)

product_embeddings_nn = embedding_tower.predict(all_inputs, batch_size=1024)

check_embedding_health(product_embeddings_nn, "Neural Embedding v2")
print("--- Neural Embedding v2 ---")
evaluate_model(lambda pid, k: recommend_neural_embedding(pid, k))

Triplets anchor==positive (groupe trop petit): 0/200000
Couverture triplets par subcategory (min/max): 1875 6091
Epoch 1/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/step - loss: 0.0525 - learning_rate: 0.0010
Epoch 2/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0098 - learning_rate: 0.0010
Epoch 3/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0062 - learning_rate: 0.0010
Epoch 4/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0038 - learning_rate: 0.0010
Epoch 5/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0023 - learning_rate: 0.0010
Epoch 6/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0015 - learning_rate: 0.0010
Epoch 7/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0011 - learning_rate: 0.0010
Epoch 8/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 8.1685e-04 - learning_rate: 0.0010
Epoch 9/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 6.2474e-04 - learning_rate: 0.0010
Epoch 10/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - los

In [65]:
# ===========================================================
# B. AUTOENCODER — logging des losses par tête + capacité augmentée
# ===========================================================

BOTTLENECK_DIM = 32  # augmenté de 16 -> plus de capacité pour porter 8 signaux

cat_inputs = []
cat_embeds = []

for col in cat_features:
    inp = layers.Input(shape=(1,), name=f"input_{col}")
    emb = layers.Embedding(
        input_dim=cat_dims[col],
        output_dim=EMBED_DIM_CAT,
        embeddings_regularizer=regularizers.l2(L2_REG),
        name=f"embed_{col}"
    )(inp)
    emb = layers.Flatten()(emb)
    cat_inputs.append(inp)
    cat_embeds.append(emb)

num_input = layers.Input(shape=(len(num_features),), name="input_numerical")

x = layers.Concatenate()(cat_embeds + [num_input])
x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)

# Bottleneck sans activity_regularizer L1 cette fois (isolé pour diagnostic)
bottleneck = layers.Dense(
    BOTTLENECK_DIM,
    activation=None,
    name="product_embedding"
)(x)

d = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(bottleneck)
d = layers.Dropout(DROPOUT_RATE)(d)
d = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(d)

outputs = []
losses = {}
loss_weights = {}
metrics = {}

for col in cat_features:
    out = layers.Dense(cat_dims[col], activation="softmax", name=f"recon_{col}")(d)
    outputs.append(out)
    losses[f"recon_{col}"] = "sparse_categorical_crossentropy"
    metrics[f"recon_{col}"] = "accuracy"
    # poids proportionnel au nb de classes, cast explicite en float Python
    # pour éviter un mismatch de dtype (float64 numpy vs float32 Keras)
    loss_weights[f"recon_{col}"] = float(np.log1p(cat_dims[col]))

num_out = layers.Dense(len(num_features), activation=None, name="recon_numerical")(d)
outputs.append(num_out)
losses["recon_numerical"] = "mse"
loss_weights["recon_numerical"] = 0.5  # moins prioritaire que le catégoriel

embedding_model = Model(inputs=cat_inputs + [num_input], outputs=bottleneck)
autoencoder = Model(inputs=cat_inputs + [num_input], outputs=outputs)
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

train_targets = {f"recon_{col}": cat_data[col][idx_train] for col in cat_features}
train_targets["recon_numerical"] = train_target
val_targets = {f"recon_{col}": cat_data[col][idx_val] for col in cat_features}
val_targets["recon_numerical"] = val_target

early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)

history_ae = autoencoder.fit(
    train_inputs, train_targets,
    validation_data=(val_inputs, val_targets),
    epochs=100, batch_size=256,
    callbacks=[early_stop, reduce_lr], verbose=1
)

# Diagnostic : accuracy de reconstruction par variable (val)
print("\n--- Accuracy de reconstruction par variable (val) ---")
for col in cat_features:
    acc_key = f"val_recon_{col}_accuracy"
    if acc_key in history_ae.history:
        print(f"{col}: {history_ae.history[acc_key][-1]:.3f}")

product_embeddings_ae = embedding_model.predict(all_inputs, batch_size=512)
check_embedding_health(product_embeddings_ae, "Autoencoder v2")
print("--- Autoencoder v2 ---")
evaluate_model(lambda pid, k: recommend_autoencoder(pid, k))

Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 24s 66ms/step - loss: 35.9188 - recon_category_accuracy: 0.5160 - recon_category_loss: 1.4900 - recon_color_accuracy: 0.1784 - recon_color_loss: 2.3805 - recon_material_accuracy: 0.4462 - recon_material_loss: 1.5486 - recon_numerical_loss: 0.8085 - recon_season_accuracy: 0.3172 - recon_season_loss: 1.5037 - recon_size_accuracy: 0.3075 - recon_size_loss: 1.6803 - recon_style_accuracy: 0.3331 - recon_style_loss: 1.7124 - recon_subcategory_accuracy: 0.1659 - recon_subcategory_loss: 2.9478 - val_loss: 30.1405 - val_recon_category_accuracy: 0.8736 - val_recon_category_loss: 0.8623 - val_recon_color_accuracy: 0.2189 - val_recon_color_loss: 2.3463 - val_recon_material_accuracy: 0.7111 - val_recon_material_loss: 1.2293 - val_recon_numerical_loss: 0.8191 - val_recon_season_accuracy: 0.4061 - val_recon_season_loss: 1.4341 - val_recon_size_accuracy: 0.3692 - val_recon_size_loss: 1.6210 - val_recon_style_accuracy: 0.4511 - val_recon_style_loss: 1.5504 - val

In [66]:
print(history_ae.history["val_recon_numerical_loss"][-1])
print(history_ae.history["recon_numerical_loss"][-1])

0.8063198924064636
0.8448784351348877


In [67]:
loss_weights["recon_numerical"] = 5.0  # au lieu de 0.5

In [68]:
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

history_ae = autoencoder.fit(
    train_inputs, train_targets,
    validation_data=(val_inputs, val_targets),
    epochs=100, batch_size=256,
    callbacks=[early_stop, reduce_lr], verbose=1
)

print("\n--- Accuracy de reconstruction par variable (val) ---")
for col in cat_features:
    acc_key = f"val_recon_{col}_accuracy"
    if acc_key in history_ae.history:
        print(f"{col}: {history_ae.history[acc_key][-1]:.3f}")

print("val_recon_numerical_loss:", history_ae.history["val_recon_numerical_loss"][-1])

product_embeddings_ae = embedding_model.predict(all_inputs, batch_size=512)
check_embedding_health(product_embeddings_ae, "Autoencoder v3")
print("--- Autoencoder v3 ---")
evaluate_model(lambda pid, k: recommend_autoencoder(pid, k))

Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 22s 62ms/step - loss: 7.7257 - recon_category_accuracy: 0.9733 - recon_category_loss: 0.0788 - recon_color_accuracy: 0.9092 - recon_color_loss: 0.2844 - recon_material_accuracy: 0.9329 - recon_material_loss: 0.1773 - recon_numerical_loss: 0.8149 - recon_season_accuracy: 0.8990 - recon_season_loss: 0.2842 - recon_size_accuracy: 0.8969 - recon_size_loss: 0.3001 - recon_style_accuracy: 0.9040 - recon_style_loss: 0.2581 - recon_subcategory_accuracy: 0.9455 - recon_subcategory_loss: 0.1595 - val_loss: 4.0605 - val_recon_category_accuracy: 0.9999 - val_recon_category_loss: 6.0918e-04 - val_recon_color_accuracy: 1.0000 - val_recon_color_loss: 0.0124 - val_recon_material_accuracy: 1.0000 - val_recon_material_loss: 0.0059 - val_recon_numerical_loss: 0.7372 - val_recon_season_accuracy: 1.0000 - val_recon_season_loss: 0.0276 - val_recon_size_accuracy: 0.9965 - val_recon_size_loss: 0.0893 - val_recon_style_accuracy: 1.0000 - val_recon_style_loss: 0.0161 - v